In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import qmc
from scipy.spatial import KDTree
from scipy import integrate

import json
import logging
from tqdm import tqdm

import ogstools as ot
import pyvista as pv

import shutil
from pathlib import Path

from meshing import create_rectangle_frac_mesh_v3
from functions_s import save_combined_mesh


# Functions and settings

In [ ]:
def sens_index(result):

    if result is None or 'values' not in result or len(result['values']) == 0:
            return 0.0, 0.0, 0.0
        
    p = np.array(result['values'])
    t = np.array(result['timevalues'])
    
    sum_c = integrate.simpson(y=p, x=t) if len(p) > 1 else 0.0
    peak = np.max(p)
    p_end = p[-1]

    return sum_c, peak, p_end


def extract_values(RUN_DIR: Path,coords: np.array): 

    pvd_files = list(Path(RUN_DIR).glob("*.pvd"))
    if not pvd_files:
        raise FileNotFoundError(f"No .pvd file found in {RUN_DIR}")
    
    pvd_path = pvd_files[0]
    ms = ot.MeshSeries(pvd_path)
    
    ms_probes=ms.probe(points=coords)
    raw_pressure_array = ms_probes['pressure']
    clean_p = np.squeeze(raw_pressure_array) #or raw_pressure_array()[:,0] taking the one point scalar

    data_bundle = {
        'values': clean_p, 
        'timevalues': np.array(ms.timevalues),
        'metadata': {
            'variable_name': 'pressure',
            'unit': 'Pa',
            'time_unit': 's',
            'coordinates': coords,
            'source_file': str(pvd_path)
        }
    }

    return data_bundle

def run_simulation_OGS(
        prj_in:Path=None,
        prj_out: Path=None,
        row:pd.Series=None,
        MESH_DIR:Path=None,
        RUN_DIR:Path=None,
        # index:int=None,
        extrac_coord:np.array=None
        ):
    try:
        temp_prj(prj_in, prj_out, row)
        model = ot.Project(input_file=prj_out,verbose=False)

        # # Build mesh arguments explicitly using repeated -m flags (NO '--')
        # domain_mesh_path = (MESH_DIR / config.DOMAIN_MESH).resolve()
        
        # # Format subdomain meshes with -m
        # subdomain_args = " ".join([f'-m "{(MESH_DIR / sub).resolve()}"' for sub in config.SUBDOMAIN_MESH])
        
        # Combine arguments
        cmd_args = f'-m "{MESH_DIR.resolve()}" -o "{RUN_DIR.resolve()}" -l error'
        
        model.run_model(
            background=False,
            args=cmd_args,
            logfile="SA_log.txt", 
            write_logs=True
        )

        # model.run_model(
        #                 background=False,
        #                 args = f'-m "{(MESH_DIR/config.DOMAIN_MESH).resolve()}" -o "{OUT_DIR.resolve()}" -- "{(MESH_DIR/config.SUBDOMAIN_MESH[0]).resolve()}" "{(MESH_DIR/config.SUBDOMAIN_MESH[1]).resolve()}"',
        #                 logfile="SA_log.txt", 
        #                 write_logs=True)

        return extract_values(RUN_DIR, extrac_coord)
    except Exception as e:
        print(f"Simulation failed with params {row}: {e}")
        return None 

def temp_prj(prj_in, prj_out, factors:dict): 

    s_xml, ds_xml = ogs_expr_ready(factors)

    model = ot.Project(input_file=prj_in,output_file=prj_out)
    medium_matrix=0
    medium=1

    xpath_s=f".//media/medium[@id='{medium}']/properties/property[name='storage']/value/expression"
    xpath_ds=f".//media/medium[@id='{medium}']/properties/property[name='storage']/dvalue/expression"
    try:
        model.replace_text(s_xml,xpath=xpath_s)
        model.replace_text(ds_xml,xpath=xpath_ds)

    except Exception as e:
        print(f"CRITICAL ERROR in PRJ update: {e}")
        raise # Stop the loop if the input file is not correctly updated

    model.write_input()

def  ogs_expr_ready(factors: dict) -> None:
    "s(p) inverted sigmoid unlinked transition"
    pst=factors['pst']
    ws= max(factors['ws'], 1e-6)
    s_plat = factors["s_plat"]
    sf0 = factors["sf0"]

    prev= "liquid_phase_pressure"

    alpha = f'(1.0 / {ws})'

    tanh_t1 = f'tanh({alpha} * ({pst} - {prev})) '
    term_A = f'(0.5 * ({sf0} - {s_plat}) * ( 1 + {tanh_t1}))'
   
    s_ab = f'({s_plat} + {term_A})'
    s_xml = s_ab 

    ds_term_A = f'(0.5 *  ({s_plat} - {sf0}) * {alpha}  * ( 1- pow({tanh_t1},2)))' 
    ds_xml = ds_term_A
       
    return s_xml, ds_xml


def log_rescale(par: dict, to_log: bool = True) -> dict:
    "to_log | True: converts to log. False: converst back from log to real"
    converted = {}
    for key, val in par.items():
        val_arr = np.asarray(val, dtype=float)
        if to_log:
            converted[key] = np.log10(val_arr)
        else:
            converted[key] = 10.0 ** val_arr
    return converted


def generate_lhoat_samples(n_seeds:int=6, delta_log:float=0.2, param_bounds:dict=None):
    """
    Generate LH-OAT sample matrix for D parameters.
    n_seeds: Number of LHS base points
    delta_log: Fractional step size for OAT perturbation
    param_bounds: Dict with (min, max) in log10 space in real space (converted internally to log10)
    """
    param_names = list(param_bounds.keys())
    D = len(param_names)

    log_bounds = {k: tuple(v) for k, v in log_rescale(param_bounds, to_log=True).items()} #getting log_bounds
    
    # Generate Latin Hypercube base points in [0, 1]
    sampler = qmc.LatinHypercube(d=D)
    base_lhs = sampler.random(n=n_seeds)

    #mapping in real log bounds
    base_log=np.zeros_like(base_lhs)
    for idx, name in enumerate(param_names):
        p_min, p_max= log_bounds[name]
        base_log[:,idx]=p_min + base_lhs[:,idx]*(p_max-p_min)
    
    all_log_samples = []
    metadata = []
    run_id=0
    
    # Perturbations
    for seed_idx in range(n_seeds):
        base_pt = base_log[seed_idx].copy()
        
        # Add the base point
        all_log_samples.append(base_pt)
        metadata.append({
                        "run_id": run_id, 
                         "seed_id": seed_idx,
                         "type": "base",
                         "varied_param": "none",
                         "step_direction0":0
        })
        run_id += 1
        
        # Perturb each parameter one-at-a-time
        for d in range(D):
            oat_pt = base_pt.copy()
            p_name=param_names[d]
            p_min, p_max=log_bounds[p_name]

            # Step up (or down if near upper boundary) 
            if oat_pt[d] + delta_log <= p_max:
                oat_pt[d] += delta_log
                direction=+1
            elif oat_pt[d]-delta_log>=p_min:
                oat_pt[d]-=delta_log
                direction=-1
            else: #control boundary
                dist_to_max=p_max-oat_pt[d]
                dist_to_min=p_min-oat_pt[d]
                if dist_to_max >= dist_to_min:
                    oat_pt[d]=p_max
                    direction=+1
                else:
                    oat_pt[d] = p_min
                    direction=-1

            oat_pt[d]=np.clip(oat_pt[d],p_min,p_max) #clip numerical float creeps
                
            all_log_samples.append(oat_pt)
            metadata.append({
                "run_id": run_id,
                "seed_id": seed_idx,
                "type": "oat",
                "varied_param": p_name,
                "step_direction":direction
            })
            run_id += 1

    meta_df=pd.DataFrame(metadata)
    meta_df["step_direction"] = meta_df["step_direction"].fillna(0).astype(int) #avoigin NaN in base

    # Log DataFrame
    df_log = pd.DataFrame(np.array(all_log_samples), columns=[f"{p}_log" for p in param_names])
    
    # Convert entire sampled matrix back to physical units 
    phys_dict = log_rescale({p: df_log[f"{p}_log"].values for p in param_names}, to_log=False)
    df_phys = pd.DataFrame(phys_dict)
    
    # Combine everything
    full_df = pd.concat([meta_df, df_phys, df_log], axis=1)
    
    return full_df


def mesh_gen(MSH_FILE=None,MESH_DIR=None,geom_data:dir=None,factors:dir=None)-> None :
    if MSH_FILE is None:
        print('Input mesh file dir.msh')
    L=factors.get('L',1.0)
    thickness=geom_data.get("h",1.0)
    r_well=geom_data.get("r_well",1.0)

    create_rectangle_frac_mesh_v3(
        MSH_FILE,
        radius= 100.0,
        height= thickness,
        mesh_size= thickness/4.0,
        center_z=-40.6,
        r_well = r_well,
        length = L,
        refine_well = thickness/20.0,  # Element size at the well
        refine_frac = thickness/30.0   # Element size along the fracture
    ) 

    meshes = ot.Meshes.from_gmsh(MSH_FILE, log=False)
    for name, mesh in meshes.items():
        vtu_path = (MESH_DIR / f"rectangle_{name}.vtu").as_posix()
        pv.save_meshio(vtu_path, mesh)
        # print(f"Saved {vtu_path}") #hiding in multiple mesh generating evaluaiton, also from function_S

    combined_vtu = (MESH_DIR / "combined_fracture_mesh.vtu").as_posix()
    save_combined_mesh(MSH_FILE, combined_vtu)


In [3]:
#preconditions: bounds in real space

cwd=Path.cwd()

OGS_PATH = r"C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin"
#OGS_PATH = None

if OGS_PATH is not None:
    os.environ["OGS_BIN_PATH"] = OGS_PATH
OUT_DIR = Path(os.environ.get("OGS_TESTRUNNER_OUT_DIR", "_out_t"))
MESH_DIR = OUT_DIR / "mesh"
RUN_DIR=OUT_DIR / "run"
RESULTS_DIR = OUT_DIR / "results"
npy_dir = RESULTS_DIR / "raw_data"

shutil.rmtree(OUT_DIR, ignore_errors=True)
MESH_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
npy_dir.mkdir(parents=True, exist_ok=True)

log_path=cwd/'simulation_run.log'
logging.basicConfig(
    filename=str(log_path),
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

tqdm.monitor_interval = 0 #hide OGS probes loading bar


prj_in=cwd/'BH10_20180718_40.6_SA_sf_lh.prj'
prj_out=RUN_DIR/'BH10_20180718_40.6_SA_sf_lh_temp.prj'


y=-40.6
r_st=0.038 
coords = np.array([[r_st, y, 1e-18]])
labels= f"OGS: r={r_st:>5.3f}, y={y}" 

#bounds real space
param_bounds={
    "L":        [0.5        ,   20.0    ], #m
    "sf0":      [5.0e-8     ,   9.0e-5  ], #Pa-1
    "s_plat":   [1.0e-10    ,   5.0e-8  ], #Pa-1
    "pst":      [3.1e6      ,   3.9e6   ], 
    "pjack":    [3.1e6      ,   3.9e6   ],
    "ws":       [0.2e6      ,   1.5e6   ]
    }

MSH_FILE = MESH_DIR / "symmetric_cylinder_3D.msh"
geom_data={"h":0.7, #mesh as in field data
            "r_well":0.038
}

factors={
         'L':       1.0,
         'sf0':    1.0e-8,
         's_plat':  1.0e-8, 
         'pst':     3.6e6,
         'pjack':   3.6e6,
         'ws':      0.25e6,
         'p1':      1.0,
         'p2':      12.0e6,
}

In [4]:
#test_sample=log_rescale(param_bounds,to_log=True)
#$$\text{Total Runs} = N_{\text{seeds}} \times (p + 1)$, p is parameters$
# Generates  sample runs N=6, p=5, ie, 6 * (5 + 1) = 36 sample runs

lhoat_manifest = generate_lhoat_samples(n_seeds=35, delta_log=20, param_bounds=param_bounds) #20 highly variation
lhoat_manifest.to_csv("lhoat_samples_manifestv16t2.csv", index=False)

# Run

In [5]:
#load and updates manifest progress

samples_file = cwd/"lhoat_samples_manifestv16t2.csv"
archive_file = RESULTS_DIR / "simulation_registerv16t2.jsonl"


# Initialize register file if missing
if not archive_file.exists():
    archive_file.touch()
    print(f"Initialized simulation register: {archive_file}")

# Load samples and identify completed/processed runs
df_X = pd.read_csv(samples_file)

processed_indices = set()
with open(archive_file, "r") as f:
    for line in f:
        try:
            entry = json.loads(line)
            processed_indices.add(entry["index"])
        except json.JSONDecodeError:
            continue

Initialized simulation register: _out_t\results\simulation_registerv16t2.jsonl


In [ ]:
#run training sampling
param_names = list(param_bounds.keys())

df_full=df_X[param_names].copy()

for run_id, sample_row in tqdm(df_full.iterrows(),total=len(df_full), desc="Simulations Progress", unit="run"): #run tracking per row with iterorows
    if run_id in processed_indices:
        continue  # Skip already registered runs
    
    # if OUT_DIR.exists(): 
    #     shutil.rmtree(OUT_DIR)
    # OUT_DIR.mkdir(parents=True, exist_ok=True)
    factors_update = df_full.iloc[run_id].to_dict()
    for name in param_names:
        factors[name] = factors_update[name] #updating dict 

    if RUN_DIR.exists(): shutil.rmtree(RUN_DIR)
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    if MESH_DIR.exists(): shutil.rmtree(MESH_DIR)
    MESH_DIR.mkdir(parents=True, exist_ok=True)
    
    mesh_gen(MSH_FILE,MESH_DIR,geom_data,factors)
    
    status = "failed"
    sum_c, peak, p_end = None, None, None
    
    try:  
        sim_status = run_simulation_OGS(
             prj_in,
             prj_out,
             factors, #updated factors from row dict
             MESH_DIR,
             RUN_DIR,
            #  node_index,
            coords
        )
        
        if sim_status:
            # Extract time-series output
            # data_bundle = extract_values(
            #     OUT_DIR=config.OUT_DIR, 
            #     index=node_index, 
            #     extrac_coord=actual_coords
            # )
            data_bundle=sim_status
            
            # Save raw data for optimization / bootstrapping
            np.save(npy_dir / f"raw_run_{run_id}.npy", data_bundle)
            
            # Extract raw metrics for sensitivity
            sum_c, peak, p_end = sens_index(data_bundle)
            status = "success"
            
            logging.info(f"Run {run_id} SUCCESS | Peak={peak:.2f} Pa | End={p_end:.2f} Pa")
        else:
            logging.warning(f"Run {run_id} FAILED (Simulation returned False)")
            
    except Exception as e:
            logging.error(f"Run {run_id} CRASHED: {str(e)}")

    # Register run result immediately (allows crash-safe resumption)
    with open(archive_file, "a") as f:
        record = {
            "index": run_id,
            "status": status,
            "sum_c": sum_c,
            "peak": peak,
            "p_end": p_end
        }
        f.write(json.dumps(record) + "\n")

Simulations Progress:   0%|          | 0/245 [00:00<?, ?run/s]


CMD: C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

 C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\ogs.exe -m "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\mesh" -o "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\run" -l debug _out_t\run\BH10_20180718_40.6_SA_sf_lh_temp.prj
['C:\\OGS_Binary\\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\\bin\\ogs.exe', '-m', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\mesh"', '-o', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\run"', '-l', 'debug', '_out_t\\run\\BH10_20180718_40.6_SA_sf_lh_temp.prj']


Simulations Progress:   0%|          | 1/245 [00:22<1:32:10, 22.66s/run]


CMD: C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

 C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\ogs.exe -m "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\mesh" -o "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\run" -l debug _out_t\run\BH10_20180718_40.6_SA_sf_lh_temp.prj
['C:\\OGS_Binary\\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\\bin\\ogs.exe', '-m', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\mesh"', '-o', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\run"', '-l', 'debug', '_out_t\\run\\BH10_20180718_40.6_SA_sf_lh_temp.prj']


Simulations Progress:   1%|          | 2/245 [01:04<2:17:33, 33.96s/run]

...

Simulations Progress:  99%|█████████▉| 243/245 [2:23:52<01:01, 30.57s/run]


CMD: C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

 C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\ogs.exe -m "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\mesh" -o "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\run" -l debug _out_t\run\BH10_20180718_40.6_SA_sf_lh_temp.prj
['C:\\OGS_Binary\\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\\bin\\ogs.exe', '-m', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\mesh"', '-o', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\run"', '-l', 'debug', '_out_t\\run\\BH10_20180718_40.6_SA_sf_lh_temp.prj']


Simulations Progress: 100%|█████████▉| 244/245 [2:24:30<00:32, 32.59s/run]


CMD: C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\NodeReordering

 C:\OGS_Binary\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\bin\ogs.exe -m "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\mesh" -o "C:\Users\penack5q\Activities\OpenGeoSys\github\Benchmarks\Line_source_SA\_out_t\run" -l debug _out_t\run\BH10_20180718_40.6_SA_sf_lh_temp.prj
['C:\\OGS_Binary\\ogs-6.5.7-Windows-10.0.26200-python-3.13.7-utils\\bin\\ogs.exe', '-m', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\mesh"', '-o', '"C:\\Users\\penack5q\\Activities\\OpenGeoSys\\github\\Benchmarks\\Line_source_SA\\_out_t\\run"', '-l', 'debug', '_out_t\\run\\BH10_20180718_40.6_SA_sf_lh_temp.prj']


Simulations Progress: 100%|██████████| 245/245 [2:25:10<00:00, 35.55s/run]
